In [ ]:
import re
from pathlib import Path
from typing import Iterable, List, Dict, Any
import pandas as pd
from openie import StanfordOpenIE


# ----------------------------
# Utilities
# ----------------------------
def sentence_split(text: str) -> List[str]:
    """Lightweight sentence splitter (no extra deps)."""
    if not text:
        return []
    t = re.sub(r"\s+", " ", str(text)).strip()
    parts = re.split(r"(?<=[.!?])\s+", t)
    return [p.strip() for p in parts if p.strip()]


def annotate_text(client: StanfordOpenIE, text: str) -> List[Dict[str, str]]:
    """Run OpenIE on a single text and return triples."""
    triples = client.annotate(text)
    return [
        {"subject": t.get("subject", ""), "relation": t.get("relation", ""), "object": t.get("object", "")}
        for t in triples
    ]


def _safe_get(df: pd.DataFrame, idx: Any, col: str) -> Any:
    """Safely extract a value; return '' if missing/NaN."""
    if col not in df.columns:
        return ""
    val = df.at[idx, col]
    if pd.isna(val):
        return ""
    return val


def dedupe_relation_longest_object(df: pd.DataFrame,
                                  subject_col: str = "subject",
                                  relation_col: str = "relation") -> pd.DataFrame:
    """Keep only one row per subject, preferring longest relation string."""
    df = df.copy()
    df["_len"] = df[relation_col].fillna("").astype(str).str.len()
    idx = df.groupby(subject_col)["_len"].idxmax()
    return df.loc[idx].drop(columns=["_len"]).reset_index(drop=True)


# ----------------------------
# Process one CSV file
# ----------------------------
def run_openie_single(
    input_csv: str,
    output_csv: str,
    text_col: str = "text",
    split_sentences: bool = False,
    max_chars: int | None = None,
    encoding_candidates: Iterable[str] = ("utf-8", "latin-1"),
    n_rows: int | None = None,   # ✅ new parameter
):
    in_path = Path(input_csv)
    out_path = Path(output_csv)

    # --- Read CSV ---
    df = None
    last_err = None
    for enc in encoding_candidates:
        try:
            df = pd.read_csv(in_path, encoding=enc, low_memory=False)
            break
        except Exception as e:
            last_err = e
    if df is None:
        print(f"[SKIP] Failed to read {in_path.name} ({last_err})")
        return

    if n_rows is not None:
        df = df.head(n_rows)  # ✅ process only first N rows
        print(f"Processing first {len(df)} rows of {in_path.name}...")

    if text_col not in df.columns:
        print(f"[SKIP] Missing column '{text_col}' in {in_path.name}")
        return

    # --- Run OpenIE ---
    rows_out: List[Dict[str, Any]] = []
    with StanfordOpenIE() as client:
        for idx, raw in df[text_col].items():
            if pd.isna(raw):
                continue
            text = str(raw).strip()
            if not text:
                continue
            if max_chars and len(text) > max_chars:
                text = text[:max_chars]

            geid = _safe_get(df, idx, "GlobalEventID")
            date_val = _safe_get(df, idx, "date")

            try:
                sentences = sentence_split(text) if split_sentences else [text]
                for sent in sentences:
                    for t in annotate_text(client, sent):
                        rows_out.append({
                            "GlobalEventID": geid,
                            "date": date_val,
                            "source_index": idx,
                            "subject": t["subject"],
                            "relation": t["relation"],
                            "object": t["object"],
                        })
            except Exception as e:
                rows_out.append({
                    "GlobalEventID": geid,
                    "date": date_val,
                    "source_index": idx,
                    "subject": "",
                    "relation": f"ERROR: {e}",
                    "object": "",
                })

    # --- Deduplicate ---
    out_cols = ["GlobalEventID", "date", "source_index", "subject", "relation", "object"]
    triples_df = pd.DataFrame(rows_out, columns=out_cols)
    triples_df = dedupe_relation_longest_object(triples_df)


    # --- Save output ---
    out_path.parent.mkdir(parents=True, exist_ok=True)
    triples_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"[OK] {in_path.name} → {out_path.name} ({len(triples_df)} rows after dedupe)")
    return triples_df


In [ ]:
run_openie_batch(
    input_dir=input_dr,
    output_dir=output_dr,
    text_col="text",        # change if your column differs
    split_sentences=True,   # optional
    max_chars=5000,
                        # optional
)


TypeError: run_openie_batch() got an unexpected keyword argument 'n_rows'

In [70]:
input_dr = '/data/elugos/regoose_CA/processed/'
output_dr = '/data/jupyter/openie_CA'

In [76]:
triples = run_openie_single(
    input_csv="/data/elugos/regoose_CA/processed/20140521.export_goosed.csv",
    output_csv="/data/elugos/regoose_CA/triples_CA/20140521_triples.csv",
    text_col="text",
    split_sentences=True,
    max_chars=5000,
    n_rows=10
)

Processing first 10 rows of 20140521.export_goosed.csv...
Starting server with command: java -Xmx8G -cp /home/jupyter/.stanfordnlp_resources/stanford-corenlp-4.5.3/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-2020098b4a11445c.props -preload openie


KeyboardInterrupt: 

In [77]:
data = [
    {"GlobalEventID": 1, "date": "2025-01-01", "text": "Marie Curie discovered radium."},
    {"GlobalEventID": 2, "date": "2025-01-02", "text": "Albert Einstein developed the theory of relativity."},
    {"GlobalEventID": 3, "date": "2025-01-03", "text": "Isaac Newton formulated the laws of motion."},
]

pd.DataFrame(data).to_csv("test_input.csv", index=False)
print("✅ Created test_input.csv")

✅ Created test_input.csv


In [78]:
triples = run_openie_single(
    input_csv="test_input.csv",
    output_csv="test_output.csv",
    text_col="text",
    split_sentences=True,
    n_rows=3
)

Processing first 3 rows of test_input.csv...
Starting server with command: java -Xmx8G -cp /home/jupyter/.stanfordnlp_resources/stanford-corenlp-4.5.3/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-69dc8cb57bd8487e.props -preload openie


KeyboardInterrupt: 

In [79]:
print("Starting StanfordOpenIE...")
with StanfordOpenIE() as client:
    triples = client.annotate("Barack Obama was born in Hawaii.")
print("Triples:", triples)

Starting StanfordOpenIE...
Starting server with command: java -Xmx8G -cp /home/jupyter/.stanfordnlp_resources/stanford-corenlp-4.5.3/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-e8a24b3220f54fdf.props -preload openie


KeyboardInterrupt: 

In [82]:
from openie import StanfordOpenIE
import time

print("Launching OpenIE client...")
start = time.time()
with StanfordOpenIE() as client:
    print("Java server ready after", round(time.time() - start, 1), "seconds")
    triples = client.annotate("Marie Curie discovered radium.")
    print(triples)


Launching OpenIE client...
Java server ready after 0.0 seconds
Starting server with command: java -Xmx8G -cp /home/jupyter/.stanfordnlp_resources/stanford-corenlp-4.5.3/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-57a3d0477210496e.props -preload openie


PermanentlyFailedException: Timed out waiting for service to come alive.